# 🧪 Lab 06 — Catalyst Learns Space 🧠🛰️

No polygons will be intersected in this notebook.

No R-tree will be summoned.

No billion-row benchmark will be harmed.

This lab is about something quieter and, architecturally, deeper:

> **Can Spark's analyzer actually see spatial meaning in the type system?**

Catalyst has always needed to answer questions such as:

```text
Does this column exist?
What type does this expression return?
Can these UNION branches share one output type?
What type does a CASE expression need?
Is this CAST legal?
```

Historically:

```text
geom: BINARY
```

gave Catalyst essentially no geospatial type information.

Now Spark 4.2 can see contracts such as:

```text
GEOMETRY(4326)
GEOMETRY(3857)
GEOMETRY(ANY)
GEOGRAPHY(4326)
```

### 🎯 Mission objectives

We will prove that:

- spatial SRIDs are visible in Spark schemas and analyzed logical plans;
- `GEOMETRY(4326)` and `GEOMETRY(3857)` are distinct analyzer-visible types;
- a `UNION` of different fixed Geometry SRIDs widens to `GEOMETRY(ANY)`;
- a `CASE` expression can require the same mixed-SRID widening;
- the analyzed plan exposes the coercion Catalyst inserted;
- the behavior is conceptually comparable to ordinary numeric widening such as `INT → BIGINT`;
- Geometry and Geography remain distinct spatial type families;
- Spark 4.2's actual spatial CAST boundary can be probed directly;
- none of this means stock Spark suddenly knows `ST_Intersects`, `ST_Distance`, or spatial indexes.

> **Evidence boundary:** this notebook demonstrates **type/analyzer intelligence**. It does not claim Catalyst has native spatial algorithm or spatial-index planning intelligence.

## 0 — Pre-flight checks 🛰️

Target environment:

```text
PySpark  4.2.0
Spark    4.2.0
Java     17+
```

Stock Spark only.

No Sedona, no spatial extensions, and deliberately tiny datasets.

In [1]:
import sys, json, struct, warnings, re


warnings.filterwarnings(
    "ignore",
    message=r"PySpark does not yet fully support pandas >= 3\.0\.0.*",
    category=FutureWarning,
)

import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    LongType,
)

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("lab-06-catalyst-learns-space")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

java_version = spark.sparkContext._jvm.java.lang.System.getProperty("java.version")

fingerprint = {
    "python": sys.version.split()[0],
    "pyspark": pyspark.__version__,
    "spark": spark.version,
    "java": java_version,
    "ansi": spark.conf.get("spark.sql.ansi.enabled"),
    "geospatial_enabled": spark.conf.get("spark.sql.geospatial.enabled"),
}

print("🚀 Runtime fingerprint")
print(json.dumps(fingerprint, indent=2))

assert pyspark.__version__ == "4.2.0"
assert spark.version == "4.2.0"
assert int(java_version.split(".")[0]) >= 17
assert fingerprint["geospatial_enabled"].lower() == "true"

print("\n✅ Catalyst observation deck online.")

🚀 Runtime fingerprint
{
  "python": "3.14.0",
  "pyspark": "4.2.0",
  "spark": "4.2.0",
  "java": "17.0.19",
  "ansi": "true",
  "geospatial_enabled": "true"
}

✅ Catalyst observation deck online.


# 1 — Give Catalyst Some Spatial Nouns 📚

We create four tiny columns:

```text
GEOMETRY(4326)
GEOMETRY(3857)
GEOMETRY(0)
GEOGRAPHY(4326)
```

The rows barely matter.

The schemas are the evidence.

If Catalyst sees these merely as binary blobs, this lab is over before lunch.

In [2]:
def wkb_point(x, y):
    return struct.pack("<BIdd", 1, 1, float(x), float(y))

g4326 = (
    spark.createDataFrame(
        [(1, wkb_point(-3.7038, 40.4168))],
        ["id", "wkb"],
    )
    .select("id", F.st_geomfromwkb("wkb", 4326).alias("geom"))
)

g3857 = (
    spark.createDataFrame(
        [(2, wkb_point(-412305.13, 4926696.67))],
        ["id", "wkb"],
    )
    .select("id", F.st_geomfromwkb("wkb", 3857).alias("geom"))
)

g0 = (
    spark.createDataFrame(
        [(3, wkb_point(10, 20))],
        ["id", "wkb"],
    )
    .select("id", F.st_geomfromwkb("wkb").alias("geom"))
)

geog4326 = (
    spark.createDataFrame(
        [(4, wkb_point(-3.7038, 40.4168))],
        ["id", "wkb"],
    )
    .select("id", F.st_geogfromwkb("wkb").alias("geog"))
)

type_evidence = {
    "geometry_4326": g4326.schema["geom"].dataType.simpleString(),
    "geometry_3857": g3857.schema["geom"].dataType.simpleString(),
    "geometry_0": g0.schema["geom"].dataType.simpleString(),
    "geography_4326": geog4326.schema["geog"].dataType.simpleString(),
}

print("📚 CATALYST'S SPATIAL NOUNS")
for name, dtype in type_evidence.items():
    print(f"  ├─ {name:<18} → {dtype}")

assert type_evidence["geometry_4326"].lower() == "geometry(4326)"
assert type_evidence["geometry_3857"].lower() == "geometry(3857)"
assert type_evidence["geometry_0"].lower() == "geometry(0)"
assert type_evidence["geography_4326"].lower() == "geography(4326)"

lab_results = {
    "types": type_evidence,
}

📚 CATALYST'S SPATIAL NOUNS
  ├─ geometry_4326      → geometry(4326)
  ├─ geometry_3857      → geometry(3857)
  ├─ geometry_0         → geometry(0)
  ├─ geography_4326     → geography(4326)


# 2 — Open the Analyzer's Skull: `EXPLAIN EXTENDED` 🧠🔬

`df.explain("extended")` shows several plan stages.

The section we care about most is:

```text
== Analyzed Logical Plan ==
```

because that is where Catalyst has resolved names and types.

We will print only that section to keep the notebook readable.

In [3]:
def query_execution_text(df):
    return df._jdf.queryExecution().toString()

def analyzed_section(df):
    text = query_execution_text(df)
    marker = "== Analyzed Logical Plan =="
    next_marker = "== Optimized Logical Plan =="
    if marker not in text:
        return text
    section = text.split(marker, 1)[1]
    if next_marker in section:
        section = section.split(next_marker, 1)[0]
    return marker + section.rstrip()

print("🧠 ANALYZED PLAN — GEOMETRY(4326)")
print(analyzed_section(g4326))

plan_4326 = analyzed_section(g4326).lower()

assert "geometry(4326)" in plan_4326, plan_4326

lab_results["analyzed_plan_sees_geometry_4326"] = True

🧠 ANALYZED PLAN — GEOMETRY(4326)
== Analyzed Logical Plan ==
id: bigint, geom: geometry(4326)
Project [id#0L, st_geomfromwkb(wkb#1, 4326) AS geom#2]
+- LogicalRDD [id#0L, wkb#1], false


## 🧬 Analyzer Control: Bytes vs Spatial Meaning

One tiny control makes the Catalyst story self-contained.

We show the analyzed plan for the **same kind of WKB payload** in two contracts:

```text
payload: BINARY
        🆚
geom: GEOMETRY(4326)
```

The point is not that Catalyst suddenly understands Madrid's shape from bytes.

The point is that after the native constructor, **spatial meaning exists in a datatype Catalyst can see**.

In [ ]:
raw_binary_control = spark.createDataFrame(
    [(1, wkb_point(-3.7038, 40.4168))],
    ["id", "payload"],
)

native_control = raw_binary_control.select(
    "id",
    F.st_geomfromwkb("payload", 4326).alias("geom"),
)

raw_control_plan = analyzed_section(raw_binary_control)
native_control_plan = analyzed_section(native_control)

print("🧬 RAW BINARY — ANALYZED PLAN")
print(raw_control_plan)

print("\n🌍 NATIVE GEOMETRY — ANALYZED PLAN")
print(native_control_plan)

raw_type = raw_binary_control.schema["payload"].dataType.simpleString()
native_type = native_control.schema["geom"].dataType.simpleString()

print("\n📡 ANALYZER CONTROL")
print(f"  ├─ raw payload  : {raw_type}")
print(f"  └─ native value : {native_type}")

assert raw_type.lower() == "binary"
assert native_type.lower() == "geometry(4326)"
assert "binary" in raw_control_plan.lower()
assert "geometry(4326)" in native_control_plan.lower()

lab_results.update({
    "binary_control_type": raw_type,
    "native_control_type": native_type,
})

That single plan already proves an important point:

> The SRID is not hidden in some Python object or application-side convention. It is visible during Spark SQL analysis.

Now let's give Catalyst a conflict to resolve.

# 3 — `UNION`: Catalyst Opens the Spatial Multiverse 🌌

We combine:

```text
left.geom  = GEOMETRY(4326)
right.geom = GEOMETRY(3857)
```

One `UNION` output column needs one datatype.

Neither fixed SRID can honestly describe every row.

So the interesting question is:

> **What common type does Catalyst choose, and can we see that choice in the analyzed plan?**

In [4]:
mixed_union = g4326.unionByName(g3857)

union_type = mixed_union.schema["geom"].dataType.simpleString()

print("🌌 UNION OUTPUT SCHEMA")
mixed_union.printSchema()

print("\n🧠 ANALYZED LOGICAL PLAN")
union_plan = analyzed_section(mixed_union)
print(union_plan)

union_rows = mixed_union.select(
    "id",
    F.expr("typeof(geom)").alias("column_type"),
    F.st_srid("geom").alias("value_srid"),
).orderBy("id").collect()

print("\n📡 RESULTING CONTRACT")
for row in union_rows:
    print(
        f"  ├─ row {row.id}: column={row.column_type}, "
        f"value SRID={row.value_srid}"
    )

assert union_type.lower() == "geometry(any)"
assert [row.value_srid for row in union_rows] == [4326, 3857]
assert "geometry(any)" in union_plan.lower(), union_plan

lab_results.update({
    "union_type": union_type,
    "union_srids": [row.value_srid for row in union_rows],
    "union_plan_has_any": "geometry(any)" in union_plan.lower(),
})

🌌 UNION OUTPUT SCHEMA
root
 |-- id: long (nullable = true)
 |-- geom: geometry(any) (nullable = true)


🧠 ANALYZED LOGICAL PLAN
== Analyzed Logical Plan ==
id: bigint, geom: geometry(any)
Union false, false
:- Project [id#0L, cast(geom#2 as geometry(any)) AS geom#12]
:  +- Project [id#0L, st_geomfromwkb(wkb#1, 4326) AS geom#2]
:     +- LogicalRDD [id#0L, wkb#1], false
+- Project [id#3L, cast(geom#5 as geometry(any)) AS geom#13]
   +- Project [id#3L, geom#5]
      +- Project [id#3L, st_geomfromwkb(wkb#4, 3857) AS geom#5]
         +- LogicalRDD [id#3L, wkb#4], false

📡 RESULTING CONTRACT
  ├─ row 1: column=geometry(any), value SRID=4326
  ├─ row 2: column=geometry(any), value SRID=3857


## What happened?

The values remained:

```text
4326
3857
```

but the common column became:

```text
GEOMETRY(ANY)
```

That is analyzer-level type widening.

Catalyst did **not** transform the coordinates.

It widened what the output schema is willing to claim.

> **Less certainty about the CRS, more honesty about the rows.**

# 4 — Numeric Widening as the Control Experiment 🔢

This sounds exotic only because the datatype is spatial.

Catalyst already does this kind of reasoning for ordinary values.

We create:

```text
INT
BIGINT
```

and union them.

Then we compare that analyzed plan with the spatial one.

The analogy is not that SRIDs behave mathematically like integers.

The analogy is:

> **Catalyst resolves incompatible branch types by finding a common output contract.**

In [5]:
ints = spark.createDataFrame(
    [(1,)],
    StructType([StructField("value", IntegerType(), False)]),
)

longs = spark.createDataFrame(
    [(2,)],
    StructType([StructField("value", LongType(), False)]),
)

numeric_union = ints.unionByName(longs)

numeric_type = numeric_union.schema["value"].dataType.simpleString()
numeric_plan = analyzed_section(numeric_union)

print("🔢 NUMERIC UNION")
print(f"  output type: {numeric_type}")
print("\n🧠 ANALYZED LOGICAL PLAN")
print(numeric_plan)

assert numeric_type.lower() == "bigint"

lab_results.update({
    "numeric_union_type": numeric_type,
    "numeric_plan_contains_cast": "cast(" in numeric_plan.lower(),
})

🔢 NUMERIC UNION
  output type: bigint

🧠 ANALYZED LOGICAL PLAN
== Analyzed Logical Plan ==
value: bigint
Union false, false
:- Project [cast(value#18 as bigint) AS value#20L]
:  +- LogicalRDD [value#18], false
+- Project [value#19L]
   +- LogicalRDD [value#19L], false


The control experiment gives us the useful mental model:

```text
UNION(INT, BIGINT)
        ↓
      BIGINT
```

and:

```text
GEOMETRY(4326) + GEOMETRY(3857)
                    ↓
             GEOMETRY(ANY)
```

Different type systems, same analyzer job:

> find a common contract that does not lie about the result.

# 5 — `CASE`: Same Problem, Different Syntax 🎭

`UNION` is not the only expression that needs one common result type.

A `CASE` expression does too.

We construct a single row containing:

```text
g4326 : GEOMETRY(4326)
g3857 : GEOMETRY(3857)
```

then ask:

```sql
CASE WHEN choose_4326
     THEN g4326
     ELSE g3857
END
```

Whatever branch runs at runtime, Catalyst must determine the output datatype **before execution**.

> **PySpark boundary note:** the two Geometry branches stay inside Spark for this experiment. Collecting a native `Geometry` into Python and passing it back through `F.lit(...)` is not a valid literal-serialization path in this runtime; that would test PySpark object marshalling, not Catalyst's spatial type coercion.

In [6]:
# Keep both native Geometry values inside Spark.
#
# IMPORTANT:
# Do NOT collect a Geometry to Python and feed it back through F.lit(...).
# PySpark 4.2 returns a Python Geometry object, but F.lit does not serialize
# that object as a JVM literal. That would fail before Catalyst gets a chance
# to analyze the CASE expression.
#
# A one-row crossJoin gives Catalyst both spatial columns directly.

case_input = (
    g4326
    .select(
        F.lit(True).alias("choose_4326"),
        F.col("geom").alias("g4326"),
    )
    .crossJoin(
        g3857.select(
            F.col("geom").alias("g3857"),
        )
    )
)

case_df = case_input.select(
    F.when(
        F.col("choose_4326"),
        F.col("g4326"),
    ).otherwise(
        F.col("g3857"),
    ).alias("chosen_geom")
)

case_type = case_df.schema["chosen_geom"].dataType.simpleString()
case_plan = analyzed_section(case_df)

print("🎭 CASE OUTPUT")
print(f"  type: {case_type}")

print("\n🧠 ANALYZED LOGICAL PLAN")
print(case_plan)

case_rows = case_df.select(
    F.expr("typeof(chosen_geom)").alias("column_type"),
    F.st_srid("chosen_geom").alias("value_srid"),
).collect()

print("\n📡 CASE RUNTIME EVIDENCE")
for row in case_rows:
    print(
        f"  ├─ column={row.column_type}, "
        f"value SRID={row.value_srid}"
    )

assert case_type.lower() == "geometry(any)", case_type
assert "geometry(any)" in case_plan.lower(), case_plan
assert all(row.column_type.lower() == "geometry(any)" for row in case_rows)
assert [row.value_srid for row in case_rows] == [4326]

lab_results.update({
    "case_type": case_type,
    "case_plan_has_any": "geometry(any)" in case_plan.lower(),
    "case_value_srids": [row.value_srid for row in case_rows],
})

🎭 CASE OUTPUT
  type: geometry(any)

🧠 ANALYZED LOGICAL PLAN
== Analyzed Logical Plan ==
chosen_geom: geometry(any)
Project [CASE WHEN choose_4326#21 THEN cast(g4326#22 as geometry(any)) ELSE cast(g3857#23 as geometry(any)) END AS chosen_geom#24]
+- Join Cross
   :- Project [true AS choose_4326#21, geom#2 AS g4326#22]
   :  +- Project [id#0L, st_geomfromwkb(wkb#1, 4326) AS geom#2]
   :     +- LogicalRDD [id#0L, wkb#1], false
   +- Project [geom#5 AS g3857#23]
      +- Project [id#3L, st_geomfromwkb(wkb#4, 3857) AS geom#5]
         +- LogicalRDD [id#3L, wkb#4], false

📡 CASE RUNTIME EVIDENCE
  ├─ column=geometry(any), value SRID=4326


# 6 — Geometry and Geography Are Different Nouns 🌍📐

Now we deliberately mix the two spatial families:

```text
GEOMETRY(4326)
GEOGRAPHY(4326)
```

Same SRID number.

Different semantics.

This is exactly where a type-aware analyzer should **not** casually pretend the values are interchangeable.

We will probe both `UNION` and explicit `CAST`.

For this section, the correct scientific approach is not to guess a cast matrix in advance.

We let this exact Spark 4.2 runtime tell us what it accepts.

In [7]:
def compact_exception(exc):
    text = str(exc).strip()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    first = lines[0] if lines else repr(exc)

    condition = None
    getter = getattr(exc, "getCondition", None)
    if callable(getter):
        try:
            condition = getter()
        except Exception:
            condition = None

    if condition is None:
        m = re.search(r"\[([A-Z0-9_.]+)\]", first)
        if m:
            condition = m.group(1)

    return type(exc).__name__, condition, first

def probe_dataframe(label, builder):
    """
    Analyzer-oriented probe.
    Accessing .schema forces Spark to analyze the expression.
    """
    try:
        df = builder()
        schema = df.schema
        return {
            "label": label,
            "accepted": True,
            "schema": schema.simpleString(),
            "error_type": None,
            "error_class": None,
            "error": None,
        }
    except Exception as exc:
        etype, condition, message = compact_exception(exc)
        return {
            "label": label,
            "accepted": False,
            "schema": None,
            "error_type": etype,
            "error_class": condition,
            "error": message,
        }

geog_as_geom_name = geog4326.select(
    "id",
    F.col("geog").alias("geom"),
)

geometry_geography_union_probe = probe_dataframe(
    "GEOMETRY(4326) UNION GEOGRAPHY(4326)",
    lambda: g4326.unionByName(geog_as_geom_name),
)

print("🌍📐 GEOMETRY vs GEOGRAPHY UNION")
print(json.dumps(geometry_geography_union_probe, indent=2))

lab_results["geometry_geography_union"] = geometry_geography_union_probe

🌍📐 GEOMETRY vs GEOGRAPHY UNION
{
  "label": "GEOMETRY(4326) UNION GEOGRAPHY(4326)",
  "accepted": false,
  "schema": null,
  "error_type": "AnalysisException",
  "error_class": "INCOMPATIBLE_COLUMN_TYPE",
  "error": "[INCOMPATIBLE_COLUMN_TYPE] UNION can only be performed on tables with compatible column types. The second column of the second table is \"GEOGRAPHY(4326)\" type which is not compatible with \"GEOMETRY(4326)\" at the same column of the first table.. SQLSTATE: 42825;"
}


# 7 — Spatial CAST Matrix: Ask Catalyst, Don't Guess 🚪

Now we probe several explicit casts.

We are interested in boundaries such as:

```text
GEOMETRY(4326) → GEOMETRY(ANY)
GEOMETRY(4326) → GEOMETRY(3857)
GEOMETRY(4326) → GEOGRAPHY(4326)
GEOMETRY(4326) → BINARY

BINARY → GEOMETRY(4326)
GEOGRAPHY(4326) → GEOGRAPHY(ANY)
GEOGRAPHY(4326) → GEOMETRY(4326)
```

This is intentionally an observational matrix.

Why?

Because the important lesson is that **Catalyst now has a spatial type system to accept or reject against**.

We record what this runtime actually does rather than turning the notebook into a collection of assumptions.

In [8]:
g4326.createOrReplaceTempView("geometry_4326")
geog4326.createOrReplaceTempView("geography_4326")

binary_df = spark.createDataFrame(
    [(1, wkb_point(-3.7038, 40.4168))],
    ["id", "payload"],
)
binary_df.createOrReplaceTempView("binary_payload")

cast_sql = [
    (
        "GEOMETRY(4326) → GEOMETRY(ANY)",
        "SELECT CAST(geom AS GEOMETRY(ANY)) AS value FROM geometry_4326",
    ),
    (
        "GEOMETRY(4326) → GEOMETRY(3857)",
        "SELECT CAST(geom AS GEOMETRY(3857)) AS value FROM geometry_4326",
    ),
    (
        "GEOMETRY(4326) → GEOGRAPHY(4326)",
        "SELECT CAST(geom AS GEOGRAPHY(4326)) AS value FROM geometry_4326",
    ),
    (
        "GEOMETRY(4326) → BINARY",
        "SELECT CAST(geom AS BINARY) AS value FROM geometry_4326",
    ),
    (
        "BINARY → GEOMETRY(4326)",
        "SELECT CAST(payload AS GEOMETRY(4326)) AS value FROM binary_payload",
    ),
    (
        "GEOGRAPHY(4326) → GEOGRAPHY(ANY)",
        "SELECT CAST(geog AS GEOGRAPHY(ANY)) AS value FROM geography_4326",
    ),
    (
        "GEOGRAPHY(4326) → GEOMETRY(4326)",
        "SELECT CAST(geog AS GEOMETRY(4326)) AS value FROM geography_4326",
    ),
]

cast_results = []

# Expected analyzer failures are evidence, not notebook emergencies.
# Silence Spark's verbose SQLQueryContextLogger while we deliberately
# probe invalid CASTs; the compact matrix below keeps the useful result.
_log4j_configurator = spark._jvm.org.apache.logging.log4j.core.config.Configurator
_log4j_level = spark._jvm.org.apache.logging.log4j.Level
for _logger_name in [
    "SQLQueryContextLogger",
    "org.apache.spark.sql.catalyst.trees.SQLQueryContextLogger",
]:
    _log4j_configurator.setLevel(_logger_name, _log4j_level.OFF)

try:
    for label, sql in cast_sql:
        try:
            df = spark.sql(sql)
            dtype = df.schema["value"].dataType.simpleString()
            result = {
                "label": label,
                "accepted": True,
                "result_type": dtype,
                "error_class": None,
                "error": None,
            }
        except Exception as exc:
            _, condition, message = compact_exception(exc)
            result = {
                "label": label,
                "accepted": False,
                "result_type": None,
                "error_class": condition,
                "error": message,
            }

        cast_results.append(result)
finally:
    for _logger_name in [
        "SQLQueryContextLogger",
        "org.apache.spark.sql.catalyst.trees.SQLQueryContextLogger",
    ]:
        _log4j_configurator.setLevel(_logger_name, _log4j_level.ERROR)

print("🚪 SPATIAL CAST MATRIX — THIS RUNTIME")
print("-" * 110)
for r in cast_results:
    status = "✅ ACCEPT" if r["accepted"] else "💥 REJECT"
    target = r["result_type"] if r["accepted"] else (r["error_class"] or r["error"][:70])
    print(f"{status:<10} | {r['label']:<45} | {target}")

lab_results["cast_results"] = cast_results

🚪 SPATIAL CAST MATRIX — THIS RUNTIME
--------------------------------------------------------------------------------------------------------------
✅ ACCEPT   | GEOMETRY(4326) → GEOMETRY(ANY)                | geometry(any)
💥 REJECT   | GEOMETRY(4326) → GEOMETRY(3857)               | DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION
✅ ACCEPT   | GEOMETRY(4326) → GEOGRAPHY(4326)              | geography(4326)
💥 REJECT   | GEOMETRY(4326) → BINARY                       | DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION
💥 REJECT   | BINARY → GEOMETRY(4326)                       | DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION
✅ ACCEPT   | GEOGRAPHY(4326) → GEOGRAPHY(ANY)              | geography(any)
✅ ACCEPT   | GEOGRAPHY(4326) → GEOMETRY(4326)              | geometry(4326)


## Why this cast probe matters

The point is **not** that every spatial type should cast to every other spatial type.

Quite the opposite.

Once spatial meaning is in the type system, Catalyst finally has enough information to say:

```text
yes, this conversion is allowed
```

or:

```text
no, these contracts are incompatible
```

With plain `BINARY`, that conversation barely exists.

# 8 — What Catalyst Still Does *Not* Know 🚧

Now for the boundary.

If Spark had become a full native spatial engine, we would expect functions such as:

```text
ST_Intersects
ST_Within
ST_Distance
```

or analyzer/physical-plan machinery for:

```text
spatial indexes
R-trees
spatial partition pruning
specialized spatial join operators
```

This lab does not claim any of that.

We perform a clean catalog probe for a few representative spatial verbs.

In [9]:
verb_names = [
    "st_intersects",
    "st_within",
    "st_distance",
    "st_transform",
]

catalog_names = {f.name.lower() for f in spark.catalog.listFunctions()}

verb_probe = {
    name: name in catalog_names
    for name in verb_names
}

print("🚧 STOCK SPARK 4.2 — REPRESENTATIVE SPATIAL VERBS")
for name, available in verb_probe.items():
    print(f"  ├─ {name:<14} registered? {available}")

assert all(not available for available in verb_probe.values())

lab_results["verb_probe"] = verb_probe

🚧 STOCK SPARK 4.2 — REPRESENTATIVE SPATIAL VERBS
  ├─ st_intersects  registered? False
  ├─ st_within      registered? False
  ├─ st_distance    registered? False
  ├─ st_transform   registered? False


# 📊 Post-Lab Analysis — Let Catalyst Write the Report

The next cell builds the verdict from this runtime's actual evidence.

The CAST matrix is intentionally generated from observed results.

If a future Spark release changes those boundaries, the notebook should report the new reality rather than preserve an old screenshot as sacred scripture.

In [10]:
from IPython.display import Markdown, display

accepted_casts = [
    r for r in lab_results["cast_results"] if r["accepted"]
]
rejected_casts = [
    r for r in lab_results["cast_results"] if not r["accepted"]
]

def cast_lines(items):
    if not items:
        return "(none)"
    lines = []
    for r in items:
        suffix = (
            r["result_type"]
            if r["accepted"]
            else (r["error_class"] or r["error"][:90])
        )
        lines.append(f"{r['label']}  →  {suffix}")
    return "\n".join(lines)

union_family = lab_results["geometry_geography_union"]
union_family_result = (
    union_family["schema"]
    if union_family["accepted"]
    else (union_family["error_class"] or union_family["error"])
)

analysis = f"""
# 📊 Post-Lab Analysis: Catalyst Has Learned the Nouns

Spark exposed four distinct spatial contracts to the analyzer:

```text
{lab_results['types']['geometry_4326']}
{lab_results['types']['geometry_3857']}
{lab_results['types']['geometry_0']}
{lab_results['types']['geography_4326']}
```

The analyzed plan for the 4326 Geometry explicitly contained its spatial type:

**{lab_results['analyzed_plan_sees_geometry_4326']}**

That is the foundational result.

### 1. Mixed SRIDs Become an Analyzer Problem, Not Application Folklore

Unioning fixed Geometry SRIDs:

```text
{lab_results['union_srids']}
```

produced:

**`{lab_results['union_type']}`**

and the analyzed logical plan itself exposed that mixed type:

**{lab_results['union_plan_has_any']}**

The row SRIDs survived unchanged.

Catalyst widened the **type contract**. It did not transform spatial coordinates.

### 2. This Looks Like Ordinary Type Reasoning

Our control union widened numeric values to:

**`{lab_results['numeric_union_type']}`**

The exact rules are different, but the analyzer's responsibility is recognizably similar:

```text
different branch types
        ↓
common output type
```

Spatial data has entered the same kind of pre-execution reasoning machinery Spark already uses for ordinary datatypes.

### 3. `CASE` Needs the Same Spatial Honesty

A `CASE` choosing between fixed-SRID geometries produced:

**`{lab_results['case_type']}`**

and that type was visible in the analyzed plan:

**{lab_results['case_plan_has_any']}**

Again: this happened during analysis, before any spatial algorithm was needed.

### 4. Geometry and Geography Remain Different Type Families

The direct Geometry/Geography union probe produced:

```text
{union_family_result}
```

Whether Spark accepts or rejects a particular conversion, the important thing is that these values reach Catalyst as distinct spatial contracts rather than undifferentiated bytes.

### 5. This Runtime's Explicit Spatial CAST Boundary

Accepted:

```text
{cast_lines(accepted_casts)}
```

Rejected:

```text
{cast_lines(rejected_casts)}
```

This matrix is runtime evidence, not a hand-written compatibility table.

### 6. The Verbs Are Still Mostly Elsewhere

Representative stock Spark functions registered:

```text
{json.dumps(lab_results['verb_probe'], indent=2)}
```

None of `ST_Intersects`, `ST_Within`, `ST_Distance`, or `ST_Transform` was registered in this stock Spark 4.2 runtime.

So the boundary is very clear:

```text
Spark 4.2
→ spatial TYPE intelligence

full spatial engine
→ spatial ALGORITHM + EXECUTION intelligence
```

> ## 🚀 Mission Verdict
> **Catalyst can now see what kind of spatial thing it is looking at.**
>
> It can distinguish Geometry from Geography, fixed SRIDs from mixed SRIDs, widen output contracts, and enforce type compatibility during analysis.
>
> That is not a spatial join engine.
>
> But an optimizer cannot reason about information the type system never exposed.
>
> **Spark has learned the nouns. Most of the verbs are still elsewhere.**
"""

display(Markdown(analysis))


# 📊 Post-Lab Analysis: Catalyst Has Learned the Nouns

Spark exposed four distinct spatial contracts to the analyzer:

```text
geometry(4326)
geometry(3857)
geometry(0)
geography(4326)
```

The analyzed plan for the 4326 Geometry explicitly contained its spatial type:

**True**

That is the foundational result.

### 1. Mixed SRIDs Become an Analyzer Problem, Not Application Folklore

Unioning fixed Geometry SRIDs:

```text
[4326, 3857]
```

produced:

**`geometry(any)`**

and the analyzed logical plan itself exposed that mixed type:

**True**

The row SRIDs survived unchanged.

Catalyst widened the **type contract**. It did not transform spatial coordinates.

### 2. This Looks Like Ordinary Type Reasoning

Our control union widened numeric values to:

**`bigint`**

The exact rules are different, but the analyzer's responsibility is recognizably similar:

```text
different branch types
        ↓
common output type
```

Spatial data has entered the same kind of pre-execution reasoning machinery Spark already uses for ordinary datatypes.

### 3. `CASE` Needs the Same Spatial Honesty

A `CASE` choosing between fixed-SRID geometries produced:

**`geometry(any)`**

and that type was visible in the analyzed plan:

**True**

Again: this happened during analysis, before any spatial algorithm was needed.

### 4. Geometry and Geography Remain Different Type Families

The direct Geometry/Geography union probe produced:

```text
INCOMPATIBLE_COLUMN_TYPE
```

Whether Spark accepts or rejects a particular conversion, the important thing is that these values reach Catalyst as distinct spatial contracts rather than undifferentiated bytes.

### 5. This Runtime's Explicit Spatial CAST Boundary

Accepted:

```text
GEOMETRY(4326) → GEOMETRY(ANY)  →  geometry(any)
GEOMETRY(4326) → GEOGRAPHY(4326)  →  geography(4326)
GEOGRAPHY(4326) → GEOGRAPHY(ANY)  →  geography(any)
GEOGRAPHY(4326) → GEOMETRY(4326)  →  geometry(4326)
```

Rejected:

```text
GEOMETRY(4326) → GEOMETRY(3857)  →  DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION
GEOMETRY(4326) → BINARY  →  DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION
BINARY → GEOMETRY(4326)  →  DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION
```

This matrix is runtime evidence, not a hand-written compatibility table.

### 6. The Verbs Are Still Mostly Elsewhere

Representative stock Spark functions registered:

```text
{
  "st_intersects": false,
  "st_within": false,
  "st_distance": false,
  "st_transform": false
}
```

None of `ST_Intersects`, `ST_Within`, `ST_Distance`, or `ST_Transform` was registered in this stock Spark 4.2 runtime.

So the boundary is very clear:

```text
Spark 4.2
→ spatial TYPE intelligence

full spatial engine
→ spatial ALGORITHM + EXECUTION intelligence
```

> ## 🚀 Mission Verdict
> **Catalyst can now see what kind of spatial thing it is looking at.**
>
> It can distinguish Geometry from Geography, fixed SRIDs from mixed SRIDs, widen output contracts, and enforce type compatibility during analysis.
>
> That is not a spatial join engine.
>
> But an optimizer cannot reason about information the type system never exposed.
>
> **Spark has learned the nouns. Most of the verbs are still elsewhere.**


## ✅ What This Run Proved

```text
GEOMETRY(4326) visible to Catalyst               ✅
GEOMETRY(3857) visible as a distinct type         ✅
GEOGRAPHY(4326) visible as a distinct family      ✅
Analyzed Logical Plan exposes spatial types       ✅
4326 + 3857 UNION → GEOMETRY(ANY)                 ✅
per-row SRIDs remain unchanged                    ✅
CASE can widen to GEOMETRY(ANY)                   ✅
numeric widening provides a useful control        ✅
spatial CAST boundaries are analyzer-visible      ✅ runtime-observed
Geometry/Geography implicit UNION rejected          ✅ runtime-proven
same-SRID Geometry↔Geography explicit CASTs allowed ✅ runtime-proven
ST_Intersects / Within / Distance absent          ✅ catalog-probed
native spatial-index planning                     ❌ not claimed
native spatial algorithms                         ❌ not claimed
```

The important result is almost philosophical:

> **Before an optimizer can become spatially clever, spatial meaning has to exist somewhere the optimizer can see it.**

Spark 4.2 has moved that meaning into the type system.

## 🧠 Catalyst Crime Board — What This Run Revealed

```text
GEOMETRY(4326) ─┐
                ├── UNION ──→ GEOMETRY(ANY)
GEOMETRY(3857) ─┘
                       │
                       └── row SRIDs stay 4326 / 3857


GEOMETRY(4326) ─┐
                ├── CASE ───→ GEOMETRY(ANY)
GEOMETRY(3857) ─┘


GEOMETRY(4326) ── implicit UNION ── GEOGRAPHY(4326)
                          │
                          └── 💥 INCOMPATIBLE_COLUMN_TYPE


GEOMETRY(4326) ── explicit CAST ──→ GEOGRAPHY(4326) ✅
GEOGRAPHY(4326) ─ explicit CAST ──→ GEOMETRY(4326)  ✅


GEOMETRY(4326) ── CAST ──→ GEOMETRY(3857) 💥
BINARY ────────── CAST ──→ GEOMETRY(4326) 💥
GEOMETRY(4326) ── CAST ──→ BINARY         💥
```

That implicit/explicit contrast is the particularly interesting part:

> **Catalyst treats Geometry and Geography as distinct contracts, yet allows some deliberate conversions when you explicitly ask for them.**

And still:

```text
ST_Intersects  ❌
ST_Within      ❌
ST_Distance    ❌
ST_Transform   ❌
```

**Nouns: yes. Verbs: mostly not yet.**

# 🛰️ Mission Handoff

Catalyst now knows:

```text
Geometry
Geography
fixed SRID
mixed SRID
compatible
incompatible
```

But ask it:

```text
Does this polygon overlap that polygon?
```

and stock Spark still looks around for adult supervision.

Next mission:

> **Where are the rest of the buttons? Spark's five native `ST_*` functions and the boundary where Sedona still takes over.** 🔘🧰

---

## 📚 Primary references

- Apache Spark 4.2 — SQL data types  
  https://spark.apache.org/docs/latest/sql-ref-datatypes.html

- Spark 4.2 — `GeometryType` / mixed-SRID acceptance  
  https://spark.apache.org/docs/4.2.0/api/scala/org/apache/spark/sql/types/GeometryType.html

- Spark 4.2 — `GeographyType` / mixed-SRID acceptance  
  https://spark.apache.org/docs/4.2.0/api/scala/org/apache/spark/sql/types/GeographyType.html

- Apache Spark 4.2 — built-in geospatial functions  
  https://spark.apache.org/docs/latest/sql-ref-functions-builtin.html

The notebook intentionally records the CAST matrix from the actual runtime instead of asserting undocumented conversions.

In [11]:
spark.stop()
print("🧠 Spark stopped. Catalyst may now forget the nouns until the next session.")

🧠 Spark stopped. Catalyst may now forget the nouns until the next session.
